In [1]:
from pathlib import Path
import gcamreader
import os
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import plotly.express as px
import pandas as pd

In [2]:
def to_Mt(row):
    val, unit = row['value'], row['Units']
    if unit == 'Tg':
        return val
    elif unit == 'Gg':
        return val * 1e-3
    elif unit == 'MTC':
        return val * (44.009 / 12.011)
    else:
        raise ValueError(f"Unknown unit: {unit}")

# AR5 100-yr GWP
GWP_AR5 = {
    'CO2':    1,
    'CH4':    28,
    'N2O':    265,
    'HFC125': 3500,
    'HFC134a':1430,
    'HFC143a':4470,
    'HFC23':  14800,
    'HFC32':  675,
    'HFC43':  1500,
    'HFC227ea':3220,
    'HFC236fa':9810,
    'SF6':    23500,
    'C2F6':   12200,
    'CF4':    6630,
}

In [3]:
proj_path = Path("/data/project/tae/gcam-core")
xml_path = proj_path / "input" / "gcamdata" / "xml"
db_path = proj_path / "output"

In [4]:
dbpath = "../output/"  # relative to current working directory
dbfile = "database_basexdb_korea_2035_20250716"
conn = gcamreader.LocalDBConn(dbpath, dbfile)
queries = gcamreader.parse_batch_query(os.path.join('..', 'output', 'queries','Main_queries.xml'))

Database scenarios: Current-Policy, Enhanced-Ambition


In [5]:
scenarios = list(conn.listScenariosInDB()['name'])
scenarios

['Current-Policy', 'Enhanced-Ambition']

In [6]:
for i, q in enumerate(queries):
    print(i, q.title)

0 primary energy consumption by region (avg fossil efficiency)
1 primary energy consumption by region (direct equivalent)
2 primary energy consumption with CCS by region (direct equivalent)
3 resource production
4 resource production by tech and vintage
5 resource supply curves
6 regional primary energy prices
7 elec gen by region (incl CHP)
8 elec gen by subsector
9 elec gen by gen tech
10 elec gen by gen tech and cooling tech
11 elec gen by gen tech and cooling tech and vintage
12 elec gen by gen tech and cooling tech (new)
13 elec energy input by subsector
14 elec energy input by elec gen tech
15 elec energy input by elec gen tech and cooling tech
16 elec prices by sector
17 elec gen costs by subsector
18 elec gen costs by tech
19 elec gen costs by cooling tech
20 elec share-weights by subsector
21 elec share-weights by tech
22 elec share-weights by cooling tech
23 elec td inputs and outputs
24 cogeneration by region
25 elec consumption by demand sector
26 elec sector water withdraw

In [7]:
i = 77
q = queries[i]
title = q.title
print(title)
df = conn.runQuery(q, scenarios=scenarios, regions=['South Korea'])
df['scenario'] = df['scenario'].str.split(',').str[0]
df

building final energy by service and fuel


,Units,scenario,region,sector,input,Year,value
0,EJ,Current-Policy,South Korea,comm cooling,delivered gas,1990,0.000386
1,EJ,Current-Policy,South Korea,comm cooling,delivered gas,2005,0.008490
2,EJ,Current-Policy,South Korea,comm cooling,delivered gas,2010,0.009812
3,EJ,Current-Policy,South Korea,comm cooling,delivered gas,2015,0.010216
4,EJ,Current-Policy,South Korea,comm cooling,delivered gas,2020,0.008970
...,...,...,...,...,...,...,...
1846,EJ,Enhanced-Ambition,South Korea,resid others modern_d9,refined liquids enduse,2015,0.007192
1847,EJ,Enhanced-Ambition,South Korea,resid others modern_d9,refined liquids enduse,2020,0.005766
1848,EJ,Enhanced-Ambition,South Korea,resid others modern_d9,refined liquids enduse,2025,0.005842
1849,EJ,Enhanced-Ambition,South Korea,resid others modern_d9,refined liquids enduse,2030,0.003647


In [8]:
df['input'].unique()

array(['delivered gas', 'elect_td_bld', 'H2 retail delivery',
       'delivered biomass', 'refined liquids enduse', 'delivered coal',
       'traditional biomass'], dtype=object)

In [9]:
df['value'] *= 23.8846

In [10]:
def cat_fuel(fuel):
    if fuel == 'delivered gas':
        return 'Gas'
    elif fuel == 'elect_td_bld':
        return 'Electricity'
    elif fuel == 'H2 retail delivery':
        return 'Hydrogen'
    elif fuel == 'delivered biomass':
        return 'Biomass'
    elif fuel == 'refined liquids enduse':
        return 'Oil'
    elif fuel == 'delivered coal':
        return 'Coal'
    elif fuel == 'traditional biomass':
        return 'Traditional Biomass'

In [11]:
df['fuel'] = df['input'].apply(cat_fuel)
df[(df['fuel'].isna())]

,Units,scenario,region,sector,input,Year,value,fuel


In [12]:
stack_order = [
    'Electricity', 'Biomass', 'Traditional Biomass', 'Gas', 'Coal', 'Oil', 'Hydrogen'
]

In [13]:
df['fuel'] = pd.Categorical(df['fuel'], categories=stack_order, ordered=True)
df = df.sort_values(by=['Year', 'fuel'])
df['fuel'].unique()

['Electricity', 'Biomass', 'Traditional Biomass', 'Gas', 'Coal', 'Oil', 'Hydrogen']
Categories (7, object): ['Electricity' < 'Biomass' < 'Traditional Biomass' < 'Gas' < 'Coal' < 'Oil' < 'Hydrogen']

In [14]:
df

,Units,scenario,region,sector,input,Year,value,fuel
8,EJ,Current-Policy,South Korea,comm cooling,elect_td_bld,1990,7.212433e-02,Electricity
35,EJ,Current-Policy,South Korea,comm heating,elect_td_bld,1990,5.485576e-02,Electricity
71,EJ,Current-Policy,South Korea,comm others,elect_td_bld,1990,9.652340e-01,Electricity
87,EJ,Current-Policy,South Korea,resid cooling modern_d1,elect_td_bld,1990,1.489104e-03,Electricity
95,EJ,Current-Policy,South Korea,resid cooling modern_d10,elect_td_bld,1990,3.531004e-02,Electricity
...,...,...,...,...,...,...,...,...
876,EJ,Current-Policy,South Korea,resid others modern_d8,H2 retail delivery,2035,4.984453e-08,Hydrogen
903,EJ,Current-Policy,South Korea,resid others modern_d9,H2 retail delivery,2035,5.413660e-08,Hydrogen
946,EJ,Enhanced-Ambition,South Korea,comm heating,H2 retail delivery,2035,7.717926e-09,Hydrogen
981,EJ,Enhanced-Ambition,South Korea,comm others,H2 retail delivery,2035,7.351513e-09,Hydrogen


In [15]:
dfAgg1 = df[(df['Year']>=2005) & (df['scenario'] == 'Current-Policy')].groupby(['Year', 'fuel'])['value'].sum().reset_index()
fig1 = px.bar(dfAgg1, x="Year", y="value", color="fuel", title="Current Policy")
dfAgg2 = df[(df['Year']>=2005) & (df['scenario'] == 'Enhanced-Ambition')].groupby(['Year', 'fuel'])['value'].sum().reset_index()
fig2 = px.bar(dfAgg2, x="Year", y="value", color="fuel", title="Enhanced Ambition")

/tmp/ipykernel_102693/2073638898.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  dfAgg1 = df[(df['Year']>=2005) & (df['scenario'] == 'Current-Policy')].groupby(['Year', 'fuel'])['value'].sum().reset_index()
/tmp/ipykernel_102693/2073638898.py:3: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [16]:
fuel_colors = {
    "Traditional Biomass": "#FFB785",    # peach
    "Oil" : "#EF0C0C",   # vivid red
    "Coal"           : "#000000",   # solid black
    "Gas"            : "#0186E0",   # bright blue
    "Biomass"        : "#099B43",   # medium green
    "Electricity"    : "#0057B5",   # deep blue
    "Hydrogen"       : "#FED30B",   # golden yellow
}

# optional: hatch / pattern overlay for the categories that are
# drawn with diagonal stripes in the figure
fuel_patterns = {

}


def bar_for(fuel, x, y):
    return go.Bar(
        name   = fuel,
        x      = x,
        y      = y,
        marker = dict(
            color   = fuel_colors[fuel],
            pattern = dict(shape = fuel_patterns.get(fuel, ""))
        )
    )


In [18]:
years = list(range(2005, 2036, 5))

# Create subplots with secondary y-axes
fig = make_subplots(
    rows=1, cols=2,
    shared_yaxes=True,
    shared_xaxes=True,
    specs=[[{"secondary_y": True}, {"secondary_y": True}]],
    subplot_titles=("Current Policy", "Enhanced Ambition")
)
# === Legend Group: Gases ===

# ------------------------------------------------------------------
# add traces from the first figure → left pane
# ------------------------------------------------------------------
for tr in fig1.data:
    tr.showlegend = False                      # keep legend single
    # NEW → colour & pattern injection
    if tr.name in fuel_colors:
        tr.marker.color = fuel_colors[tr.name]
        pattern = fuel_patterns.get(tr.name)
        if pattern:
            # marker.pattern is available from Plotly 5.3+
            tr.marker.pattern = dict(shape=pattern)

    fig.add_trace(tr, row=1, col=1, secondary_y=False)

# ------------------------------------------------------------------
# add traces from the second figure → right pane
# ------------------------------------------------------------------
for tr in fig2.data:
    if tr.name in fuel_colors:
        tr.marker.color = fuel_colors[tr.name]
        pattern = fuel_patterns.get(tr.name)
        if pattern:
            tr.marker.pattern = dict(shape=pattern)

    fig.add_trace(tr, row=1, col=2, secondary_y=False)


fig.update_layout(
    yaxis=dict(title="EJ", showgrid=True),
    yaxis1=dict(title="EJ", showgrid=True, title_font_size=20),
    
    # Hide secondary y-axis for col 1 (still used internally)
    yaxis2=dict(
        showticklabels=False,
        showgrid=False,
        zeroline=False,
        showline=False,
        title='',
        overlaying='y',
        side='right',
        range=[0, 100]
    ),
    
    # Show secondary y-axis for col 2
    yaxis4=dict(
        showticklabels=False,
        showgrid=False,
        zeroline=False,
        showline=False,
        title='',
        overlaying='y',
        side='right',
        range=[0, 100]
    ),
    barmode='stack',
    plot_bgcolor='rgba(0,0,0,0)',
    width=800, height=700,
)


fig.update_xaxes(tickangle=45)

fig.update_yaxes(range=[None, 60])

fig.update_layout(
    yaxis=dict(title="Mtoe", showgrid=True, gridcolor='lightgray'),
    yaxis3=dict(showgrid=True, gridcolor='lightgray'),
    title=dict(
        text="<b>Final energy consumption by fuel - Blds</b>",
        font=dict(size=28),
        x=0.5
    ),
    legend=dict(
        traceorder="reversed",
        font=dict(size=20),
        x=1.02, y=1,
        borderwidth=0
    )
)
fig.update_xaxes(
    tickvals=years,
    ticktext=[str(y) for y in years]
)

# adjust axis labels and ticks
fig.update_xaxes(title_font=dict(size=18), tickfont=dict(size=18))
fig.update_yaxes(title_font=dict(size=18), tickfont=dict(size=18))

# bump the subplot titles
fig.update_annotations(font=dict(size=21))

# (if you also set a main title earlier)
fig.update_layout(title=dict(font=dict(size=28)))
fig

In [48]:
i = 87
q = queries[i]
title = q.title
print(title)
df = conn.runQuery(q, scenarios=scenarios, regions=['South Korea'])
df['scenario'] = df['scenario'].str.split(',').str[0]
df

building service density by energy service


,Units,scenario,region,gcam-consumer,nodeInput,building,input,Year,value
0,GJ/m^2,Current-Policy,South Korea,comm,comm,comm_building,comm cooling,1990,0.100531
1,GJ/m^2,Current-Policy,South Korea,comm,comm,comm_building,comm cooling,2005,0.143479
2,GJ/m^2,Current-Policy,South Korea,comm,comm,comm_building,comm cooling,2010,0.148211
3,GJ/m^2,Current-Policy,South Korea,comm,comm,comm_building,comm cooling,2015,0.149012
4,GJ/m^2,Current-Policy,South Korea,comm,comm,comm_building,comm cooling,2020,0.149975
...,...,...,...,...,...,...,...,...,...
523,GJ/m^2,Enhanced-Ambition,South Korea,resid_d9,resid,resid_building,resid others modern_d9,2015,0.271421
524,GJ/m^2,Enhanced-Ambition,South Korea,resid_d9,resid,resid_building,resid others modern_d9,2020,0.266338
525,GJ/m^2,Enhanced-Ambition,South Korea,resid_d9,resid,resid_building,resid others modern_d9,2025,0.278450
526,GJ/m^2,Enhanced-Ambition,South Korea,resid_d9,resid,resid_building,resid others modern_d9,2030,0.284438


In [50]:
i = 70
q = queries[i]
title = q.title
print(title)
df = conn.runQuery(q, scenarios=scenarios, regions=['South Korea'])
df['scenario'] = df['scenario'].str.split(',').str[0]
df

building floorspace


,Units,scenario,region,building,nodeInput,building-node-input,Year,value
0,billion m^2,Current-Policy,South Korea,comm,comm,comm_building,1975,0.333635
1,billion m^2,Current-Policy,South Korea,comm,comm,comm_building,1990,0.489230
2,billion m^2,Current-Policy,South Korea,comm,comm,comm_building,2005,0.677551
3,billion m^2,Current-Policy,South Korea,comm,comm,comm_building,2010,0.689301
4,billion m^2,Current-Policy,South Korea,comm,comm,comm_building,2015,0.707074
...,...,...,...,...,...,...,...,...
479,billion m^2,Enhanced-Ambition,South Korea,resid_d9,resid,resid_building,2080,0.132639
480,billion m^2,Enhanced-Ambition,South Korea,resid_d9,resid,resid_building,2085,0.132639
481,billion m^2,Enhanced-Ambition,South Korea,resid_d9,resid,resid_building,2090,0.132639
482,billion m^2,Enhanced-Ambition,South Korea,resid_d9,resid,resid_building,2095,0.132639


In [52]:
df[(df['Year'] <= 2035)].groupby(['scenario', 'Year'])['value'].sum()

scenario           Year
Current-Policy     1975    0.510855
                   1990    1.014353
                   2005    1.735202
                   2010    1.808164
                   2015    1.887149
                   2020    2.019270
                   2025    2.080981
                   2030    2.123717
                   2035    2.153383
Enhanced-Ambition  1975    0.510855
                   1990    1.014353
                   2005    1.735202
                   2010    1.808164
                   2015    1.887149
                   2020    2.019270
                   2025    2.080981
                   2030    2.123717
                   2035    2.153383
Name: value, dtype: float64